# Map keyframe → scene

Notebook độc lập, chạy **sau** pipeline chính. Không nạp model, không đụng GPU — chỉ đọc
metadata JSON đã sinh ra rồi gán mỗi keyframe vào scene chứa nó.

**Input** (cùng cây key cho cả local lẫn S3):

```
Keyframes_<group>/keyframes/<video>/keyframes.json   [{frame, timestamp, keyframe_url}, ...]
Keyframes_<group>/keyframes/<video>/shots.csv        shot_id,start_frame,end_frame,start_ts,end_ts,kf0..kfN
Keyscence_<group>/keyscence/<video>/scenes.json      [{scene_id, start_frame, end_frame, start_time, end_time, scene_url, script}, ...]
```

**Output**:

```
<MAP_ROOT>/Maps_<group>/keyframe_scene_map.json   1 file / group
<MAP_ROOT>/keyframe_scene_map.json                gộp tất cả group (MERGE_ALL)
<MAP_ROOT>/keyframe_scene_flat.csv                1 dòng / keyframe, để nạp làm metadata FAISS
```

**Quy tắc khớp**, xét theo thứ tự, dừng ở cái đầu tiên trúng:

1. `frame` — `scene.start_frame <= kf.frame <= scene.end_frame`. Đây là đường đi đúng của
   ~100% keyframe: scene được ghép từ chính các shot sinh ra keyframe, nên biên trùng khít
   theo frame index. Tra bằng `bisect` trên mảng `start_frame` đã sort, O(log n).
2. `time` — `scene.start_time <= kf.timestamp <= scene.end_time`. Chỉ cứu trường hợp
   `frame` lệch do fps lẻ (`round(i/fps, 3)` làm tròn 3 chữ số).
3. `nearest` — keyframe rơi vào khe giữa 2 scene liền kề, gán scene gần hơn theo khoảng
   cách frame. **Nếu cell kiểm tra báo có `nearest` thì dữ liệu đang sai**, không phải
   chuyện bình thường: xem lại video đó chứ đừng bỏ qua.

`timestamp` chỉ dùng ở tầng 2 nên fps không cần biết — dữ liệu đã tự mang theo.

In [ ]:
import csv
import io
import json
import os
import re
import time
from bisect import bisect_right
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

# ── Nguồn dữ liệu ────────────────────────────────────────────────────
# "local" = đọc dưới OUT_ROOT. Chỉ dùng được khi pipeline chạy với
#           DELETE_LOCAL_AFTER_UPLOAD=False và cùng session.
# "s3"    = tải json từ bucket. Mặc định là cái này: pipeline xoá local sau upload,
#           và dữ liệu thường do nhiều người / nhiều session gộp lại.
SOURCE = "s3"

AWS_REGION      = "ap-southeast-1"
AWS_BUCKET_NAME = "aic-bucket-2026"
AWS_ACCESS_KEY=
AWS_SECRET_KEY=
try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    AWS_ACCESS_KEY = AWS_ACCESS_KEY or _sec.get_secret("AWS_ACCESS_KEY")
    AWS_SECRET_KEY = AWS_SECRET_KEY or _sec.get_secret("AWS_SECRET_KEY")
except Exception as ex:
    print(f"! Chưa lấy được Kaggle Secrets ({type(ex).__name__}) — điền tay nếu SOURCE='s3'")

S3_URL_BASE = f"https://{AWS_BUCKET_NAME}.s3.{AWS_REGION}.amazonaws.com"

OUT_ROOT = "/Users/kinh.nvnamitech.io/Documents/rackfocus/out"    # cây output của pipeline (dùng khi SOURCE="local")
MAP_ROOT = "/Users/kinh.nvnamitech.io/Documents/rackfocus/maps"   # nơi ghi file map

# ── Phạm vi ──────────────────────────────────────────────────────────
GROUPS = []        # [] = mọi group tìm thấy. vd ["L21_a", "L22_a"]
LIMIT  = 0         # 0 = tất cả. Đặt 2-3 khi chạy thử.

# ── Nội dung map ─────────────────────────────────────────────────────
USE_SHOTS_CSV  = True    # đọc thêm shots.csv -> gắn shot_id cho từng keyframe (+1 request/video)
INCLUDE_SCRIPT = False   # nhúng transcript của scene vào map. Bật -> file phình vài lần,
                         # chỉ nên bật nếu muốn map tự đủ dùng, không phải join lại scenes.json
MERGE_ALL      = True    # ghi thêm 1 file gộp mọi group
WRITE_CSV      = True    # ghi bản phẳng 1 dòng/keyframe

# ── Chạy ─────────────────────────────────────────────────────────────
MAX_WORKERS = 16      # số video đọc song song (thuần network IO, không phải CPU)
UPLOAD_MAP  = False   # đẩy Maps_<group>/ lên S3 sau khi ghi xong

os.environ.update(
    AWS_ACCESS_KEY=AWS_ACCESS_KEY or "",
    AWS_SECRET_KEY=AWS_SECRET_KEY or "",
    AWS_REGION=AWS_REGION,
    AWS_BUCKET_NAME=AWS_BUCKET_NAME,
)

print(f"source   = {SOURCE}" + (f" ({AWS_BUCKET_NAME})" if SOURCE == "s3" else f" ({OUT_ROOT})"))
print(f"map      -> {MAP_ROOT}")
print(f"group    = {GROUPS or 'tất cả'}, limit = {LIMIT or 'tất cả'}")

! Chưa lấy được Kaggle Secrets (ModuleNotFoundError) — điền tay nếu SOURCE='s3'
source   = s3 (aic-bucket-2026)
map      -> /Users/kinh.nvnamitech.io/Documents/rackfocus/maps
group    = tất cả, limit = tất cả


## Đường dẫn

Key tương đối (so với `OUT_ROOT` khi local, so với gốc bucket khi S3) là **cùng một chuỗi**
ở cả hai nguồn — pipeline upload mirror y nguyên cây. Nhờ vậy reader chỉ cần đổi cách mở
file, mọi chỗ khác dùng chung một hàm sinh key.

In [17]:
KF_PREFIX, KF_SUB   = "Keyframes_", "keyframes"
SC_PREFIX, SC_SUB   = "Keyscence_", "keyscence"   # giữ đúng chính tả của pipeline
MAP_PREFIX, MAP_SUB = "Maps_", "maps"


def kf_key(group, video="", fname=""):
    return "/".join(p for p in (f"{KF_PREFIX}{group}", KF_SUB, video, fname) if p)


def sc_key(group, video="", fname=""):
    return "/".join(p for p in (f"{SC_PREFIX}{group}", SC_SUB, video, fname) if p)


def map_key(group, fname="keyframe_scene_map.json"):
    return "/".join(p for p in (f"{MAP_PREFIX}{group}", MAP_SUB, fname) if p)


def url_of(key):
    return f"{S3_URL_BASE}/{key}"


def natkey(s):
    """Sort tự nhiên: L9_a đứng TRƯỚC L30_a."""
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def dump_json(path, data):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return path

## Reader

Hai reader cùng interface: `list_groups()`, `list_videos(group)`, `read_text(key)`.
Phần build map bên dưới chỉ nói chuyện qua interface này nên không biết dữ liệu đến từ đâu.

`list_videos` trả cả video **thiếu** một trong hai file, để báo ra thay vì lặng lẽ bỏ qua —
video có `keyframes.json` mà không có `scenes.json` nghĩa là pipeline đứt giữa chừng ở
video đó, cần chạy lại chứ không phải bỏ.

In [18]:
import importlib
import subprocess
import sys


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except ImportError:
        return False


def _pip(*args):
    print(f"  cài: {' '.join(args)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)


# (tên module để import, tên package để pip). Chỉ cài cái thiếu -> rerun nhanh.
for mod, pkg in (
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("PIL", "pillow"),
    ("onnxruntime", "onnxruntime-gpu"),
    ("transformers", "transformers>=4.37,<6"),
    ("boto3", "boto3"),
):
    if _has(mod):
        print(f"  có sẵn: {mod}")
    else:
        _pip(pkg)

  cài: numpy
  cài: pandas
  cài: pyarrow
  cài: pillow
  cài: onnxruntime-gpu
  cài: transformers>=4.37,<6
  có sẵn: boto3


/Users/kinh.nvnamitech.io/Documents/rackfocus/.venv/bin/python: No module named pip
/Users/kinh.nvnamitech.io/Documents/rackfocus/.venv/bin/python: No module named pip
/Users/kinh.nvnamitech.io/Documents/rackfocus/.venv/bin/python: No module named pip
/Users/kinh.nvnamitech.io/Documents/rackfocus/.venv/bin/python: No module named pip
/Users/kinh.nvnamitech.io/Documents/rackfocus/.venv/bin/python: No module named pip
/Users/kinh.nvnamitech.io/Documents/rackfocus/.venv/bin/python: No module named pip


In [19]:
class LocalReader:
    """Đọc từ cây thư mục OUT_ROOT."""

    def __init__(self, root=None):
        self.root = root or OUT_ROOT

    def path(self, key):
        return os.path.join(self.root, key.replace("/", os.sep))

    def list_groups(self):
        out = []
        for name in os.listdir(self.root) if os.path.isdir(self.root) else []:
            if name.startswith(KF_PREFIX) and os.path.isdir(os.path.join(self.root, name)):
                out.append(name[len(KF_PREFIX):])
        return sorted(out, key=natkey)

    def _subdirs(self, key):
        p = self.path(key)
        return {d for d in os.listdir(p)} if os.path.isdir(p) else set()

    def list_videos(self, group):
        kf = {v for v in self._subdirs(kf_key(group))
              if os.path.exists(self.path(kf_key(group, v, "keyframes.json")))}
        sc = {v for v in self._subdirs(sc_key(group))
              if os.path.exists(self.path(sc_key(group, v, "scenes.json")))}
        return sorted(kf & sc, key=natkey), sorted(kf - sc), sorted(sc - kf)

    def read_text(self, key):
        with open(self.path(key), encoding="utf-8") as f:
            return f.read()


class S3Reader:
    """Đọc trực tiếp từ bucket, không tải file xuống đĩa."""

    def __init__(self, bucket=None, region=None):
        import boto3

        self.bucket = bucket or os.environ["AWS_BUCKET_NAME"]
        ak, sk = os.environ.get("AWS_ACCESS_KEY"), os.environ.get("AWS_SECRET_KEY")
        region = region or os.environ.get("AWS_REGION")
        # boto3 client an toàn khi gọi song song từ nhiều thread.
        self.client = (boto3.client("s3", aws_access_key_id=ak, aws_secret_access_key=sk,
                                    region_name=region)
                       if ak and sk else boto3.client("s3", region_name=region))

    def list_groups(self):
        out = []
        for page in self.client.get_paginator("list_objects_v2").paginate(
                Bucket=self.bucket, Delimiter="/"):
            for cp in page.get("CommonPrefixes", []):
                p = cp["Prefix"].rstrip("/")
                if p.startswith(KF_PREFIX):
                    out.append(p[len(KF_PREFIX):])
        return sorted(out, key=natkey)

    def _videos_with(self, prefix, fname):
        """{video} có <prefix><video>/<fname>. 1 lần list phân trang, không head từng video."""
        found = set()
        for page in self.client.get_paginator("list_objects_v2").paginate(
                Bucket=self.bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                rel = obj["Key"][len(prefix):].split("/")
                if len(rel) == 2 and rel[1] == fname:
                    found.add(rel[0])
        return found

    def list_videos(self, group):
        kf = self._videos_with(kf_key(group) + "/", "keyframes.json")
        sc = self._videos_with(sc_key(group) + "/", "scenes.json")
        return sorted(kf & sc, key=natkey), sorted(kf - sc), sorted(sc - kf)

    def read_text(self, key):
        obj = self.client.get_object(Bucket=self.bucket, Key=key)
        return obj["Body"].read().decode("utf-8")


def make_reader(source=None):
    source = (source or SOURCE).lower()
    if source == "local":
        return LocalReader()
    if source == "s3":
        return S3Reader()
    raise ValueError(f"SOURCE phải là 'local' hoặc 's3', đang là {source!r}")


reader = make_reader()
_avail = reader.list_groups()
print(f"{len(_avail)} group: {_avail}")

_unknown = [g for g in GROUPS if g not in _avail]
if _unknown:
    raise ValueError(f"Group không tồn tại: {_unknown}\nCó sẵn: {_avail}")
groups_used = [g for g in _avail if g in set(GROUPS)] if GROUPS else _avail
print(f"chạy trên {len(groups_used)} group: {groups_used}")

13 group: ['L21_a', 'L22_a', 'L23_a', 'L24_a', 'L26_a', 'L26_b', 'L26_c', 'L26_d', 'L26_e', 'L27_a', 'L28_a', 'L29_a', 'L30_a']
chạy trên 13 group: ['L21_a', 'L22_a', 'L23_a', 'L24_a', 'L26_a', 'L26_b', 'L26_c', 'L26_d', 'L26_e', 'L27_a', 'L28_a', 'L29_a', 'L30_a']


## Khớp keyframe → scene

`match_scene` chỉ xét scene `i` (scene cuối cùng có `start_frame <= frame`) và scene `i+1`.
Không cần quét toàn bộ: mảng đã sort nên nếu frame rơi vào khe thì scene gần nhất chắc chắn
là một trong hai cái kề khe đó. Giữ O(1) sau bisect, không để fallback biến thành O(K·S).

In [20]:
def build_scene_index(scenes):
    """(starts, scenes) đã sort theo start_frame. scene_id gốc được giữ trong từng dict."""
    ordered = sorted(scenes, key=lambda s: (int(s["start_frame"]), int(s["end_frame"])))
    return [int(s["start_frame"]) for s in ordered], ordered


def _gap(frame, scene):
    """Khoảng cách frame tới scene, 0 nếu nằm trong."""
    s, e = int(scene["start_frame"]), int(scene["end_frame"])
    return 0 if s <= frame <= e else min(abs(frame - s), abs(frame - e))


def match_scene(frame, timestamp, starts, scenes):
    """Trả (chỉ số scene trong `scenes`, cách khớp). (None, "none") nếu video không có scene."""
    if not scenes:
        return None, "none"

    i = bisect_right(starts, frame) - 1

    # 1) frame nằm trọn trong scene — đường đi bình thường
    if 0 <= i < len(scenes) and frame <= int(scenes[i]["end_frame"]):
        return i, "frame"

    # 2) theo thời gian, cứu sai lệch làm tròn timestamp
    if timestamp is not None:
        for j in (i, i + 1):
            if 0 <= j < len(scenes):
                if float(scenes[j]["start_time"]) <= timestamp <= float(scenes[j]["end_time"]):
                    return j, "time"

    # 3) scene kề gần nhất
    cands = [j for j in (i, i + 1) if 0 <= j < len(scenes)] or [0]
    return min(cands, key=lambda j: _gap(frame, scenes[j])), "nearest"


def shot_of_frame(csv_text):
    """{frame: shot_id} từ shots.csv. Frame trùng ở 2 shot -> lấy shot_id nhỏ hơn."""
    rdr = csv.DictReader(io.StringIO(csv_text))
    cols = [c for c in (rdr.fieldnames or []) if re.fullmatch(r"kf\d+", c)]
    out = {}
    for row in rdr:
        sid = int(row["shot_id"])
        for c in cols:
            f = int(row[c])
            if f not in out or sid < out[f]:
                out[f] = sid
    return out

## Build map cho 1 video

Map ghi cả hai chiều nhưng không nhân đôi dữ liệu: `keyframes[]` mang `scene_id`,
`scenes[]` mang danh sách `frame`. Tra xuôi hay ngược đều O(1) sau khi nạp.

In [21]:
def build_video_map(group, video, keyframes, scenes, frame2shot=None,
                    include_script=None):
    """Trả dict map của 1 video. Không đụng IO — dễ test bằng dữ liệu bịa."""
    include_script = INCLUDE_SCRIPT if include_script is None else include_script
    starts, ordered = build_scene_index(scenes)

    scene_out = []
    for idx, sc in enumerate(ordered):
        sid = int(sc.get("scene_id", idx))
        row = {
            "scene_id": sid,
            "scene_file": f"scene_{sid:03d}.mp4",
            "scene_url": sc.get("scene_url") or url_of(sc_key(group, video, f"scene_{sid:03d}.mp4")),
            "start_frame": int(sc["start_frame"]),
            "end_frame": int(sc["end_frame"]),
            "start_time": float(sc["start_time"]),
            "end_time": float(sc["end_time"]),
            "keyframes": [],          # điền ở vòng dưới
        }
        if include_script:
            row["script"] = sc.get("script", "")
        scene_out.append(row)

    kf_out, counts = [], {}
    for kf in sorted(keyframes, key=lambda k: int(k["frame"])):
        frame = int(kf["frame"])
        ts = kf.get("timestamp")
        ts = float(ts) if ts is not None else None

        j, how = match_scene(frame, ts, starts, ordered)
        counts[how] = counts.get(how, 0) + 1
        sid = scene_out[j]["scene_id"] if j is not None else None
        if j is not None:
            scene_out[j]["keyframes"].append(frame)

        kf_out.append({
            "keyframe_id": f"{video}#{frame:06d}",
            "frame": frame,
            "timestamp": ts,
            "keyframe_file": f"{frame:06d}.webp",
            "keyframe_url": kf.get("keyframe_url") or url_of(kf_key(group, video, f"{frame:06d}.webp")),
            "shot_id": (frame2shot or {}).get(frame),
            "scene_id": sid,
            "match": how,
        })

    for row in scene_out:
        row["num_keyframes"] = len(row["keyframes"])

    return {
        "video_id": video,
        "group": group,
        "num_keyframes": len(kf_out),
        "num_scenes": len(scene_out),
        "match_counts": counts,
        "empty_scenes": [r["scene_id"] for r in scene_out if not r["keyframes"]],
        "scenes": scene_out,
        "keyframes": kf_out,
    }


def load_video_map(reader, group, video, use_shots_csv=None):
    """Đọc json (+csv) của 1 video rồi build map. Chạy trong thread pool."""
    use_shots_csv = USE_SHOTS_CSV if use_shots_csv is None else use_shots_csv

    keyframes = json.loads(reader.read_text(kf_key(group, video, "keyframes.json")))
    scenes = json.loads(reader.read_text(sc_key(group, video, "scenes.json")))

    frame2shot = None
    if use_shots_csv:
        try:
            frame2shot = shot_of_frame(reader.read_text(kf_key(group, video, "shots.csv")))
        except Exception as ex:
            # shots.csv chỉ để làm giàu thêm, thiếu thì shot_id = null chứ không hỏng map.
            print(f"    ! {video}: không đọc được shots.csv ({type(ex).__name__}) -> shot_id=null")

    return build_video_map(group, video, keyframes, scenes, frame2shot)

## Chạy

Song song ở mức video. Đây là IO thuần (mỗi video 2-3 request nhỏ), nên thread pool
ăn đứt vòng lặp tuần tự — GIL không phải nút thắt ở đây.

In [22]:
maps = {}        # {group: {video: map}}
errors = []      # [(group, video, lỗi)]
missing = []     # [(group, video, thiếu gì)]

t_all = time.time()
for group in groups_used:
    videos, no_scene, no_kf = reader.list_videos(group)
    for v in no_scene:
        missing.append((group, v, "thiếu scenes.json"))
    for v in no_kf:
        missing.append((group, v, "thiếu keyframes.json"))
    if LIMIT:
        videos = videos[:LIMIT]

    t0 = time.time()
    got = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(load_video_map, reader, group, v): v for v in videos}
        for fut in as_completed(futs):
            v = futs[fut]
            try:
                got[v] = fut.result()
            except Exception as err:
                errors.append((group, v, repr(err)))

    maps[group] = {v: got[v] for v in sorted(got, key=natkey)}
    n_kf = sum(m["num_keyframes"] for m in got.values())
    n_sc = sum(m["num_scenes"] for m in got.values())
    print(f"  {group:12s} {len(got):4d} video  {n_kf:7d} keyframe  {n_sc:6d} scene  "
          f"({time.time() - t0:.1f}s)"
          + (f"  ! {len(no_scene) + len(no_kf)} video thiếu file" if (no_scene or no_kf) else ""))

total_kf = sum(m["num_keyframes"] for g in maps.values() for m in g.values())
total_sc = sum(m["num_scenes"] for g in maps.values() for m in g.values())
total_v = sum(len(g) for g in maps.values())
print(f"\n{total_v} video, {total_kf} keyframe, {total_sc} scene ({time.time() - t_all:.1f}s)")

if missing:
    print(f"\n! {len(missing)} video thiếu file — pipeline đứt giữa chừng ở đó, nên chạy lại:")
    for g, v, why in missing[:20]:
        print(f"    {g}/{v}: {why}")
    if len(missing) > 20:
        print(f"    ... còn {len(missing) - 20}")

if errors:
    print(f"\n! {len(errors)} video lỗi:")
    for g, v, err in errors[:20]:
        print(f"    {g}/{v}: {err}")

  L21_a          29 video    57526 keyframe    2266 scene  (3.5s)
  L22_a          31 video    70629 keyframe    1467 scene  (4.4s)
  L23_a          25 video     5153 keyframe     152 scene  (0.7s)
  L24_a          43 video    16236 keyframe     163 scene  (1.2s)
  L26_a          99 video    58902 keyframe     573 scene  (3.6s)
  L26_b         100 video    65830 keyframe    3401 scene  (3.6s)
  L26_c          53 video    38605 keyframe    2988 scene  (2.6s)
  L26_d         100 video    73683 keyframe     953 scene  (3.8s)
  L26_e          99 video    73790 keyframe    4819 scene  (3.9s)
  L27_a          16 video    18865 keyframe    1984 scene  (0.8s)
  L28_a          24 video    36137 keyframe    2159 scene  (1.8s)
  L29_a          23 video    33275 keyframe    1023 scene  (2.6s)
  L30_a          96 video    29897 keyframe    1308 scene  (2.3s)

738 video, 578528 keyframe, 23256 scene (191.3s)


## Kiểm tra

Chạy trước khi ghi file. Ba thứ phải đúng, còn lại là cảnh báo:

- mọi keyframe có `scene_id`
- không có `match="nearest"` — có nghĩa keyframe rơi ngoài mọi scene
- `frame` của keyframe nằm trong `[start_frame, end_frame]` của scene được gán

`empty_scenes` (scene không giữ keyframe nào) chỉ là cảnh báo: một scene ngắn vẫn có shot,
mà mỗi shot luôn sinh keyframe, nên con số này lẽ ra bằng 0 — khác 0 thì đáng nhìn lại,
nhưng không chặn việc ghi file.

In [23]:
all_maps = [m for g in maps.values() for m in g.values()]

agg, unmatched, out_of_range, empty_sc = {}, [], [], []
for m in all_maps:
    for how, n in m["match_counts"].items():
        agg[how] = agg.get(how, 0) + n
    by_id = {s["scene_id"]: s for s in m["scenes"]}
    for kf in m["keyframes"]:
        if kf["scene_id"] is None:
            unmatched.append((m["video_id"], kf["frame"]))
            continue
        sc = by_id[kf["scene_id"]]
        if not sc["start_frame"] <= kf["frame"] <= sc["end_frame"]:
            out_of_range.append((m["video_id"], kf["frame"], kf["scene_id"],
                                 sc["start_frame"], sc["end_frame"]))
    empty_sc += [(m["video_id"], sid) for sid in m["empty_scenes"]]

print("cách khớp:")
for how in ("frame", "time", "nearest", "none"):
    if how in agg:
        print(f"  {how:8s} {agg[how]:8d}  ({agg[how] / max(total_kf, 1) * 100:.2f}%)")

print(f"\nkeyframe không có scene : {len(unmatched)}")
print(f"keyframe ngoài range    : {len(out_of_range)}")
print(f"scene rỗng              : {len(empty_sc)}")

for v, f, sid, s, e in out_of_range[:5]:
    print(f"    {v} frame {f} -> scene {sid} [{s}, {e}]")
for v, sid in empty_sc[:5]:
    print(f"    scene rỗng: {v} scene {sid}")

# Chỉ chặn ở cái thật sự sai. Scene rỗng để cảnh báo.
assert not unmatched, f"{len(unmatched)} keyframe không gán được scene"
assert not out_of_range, f"{len(out_of_range)} keyframe nằm ngoài range scene được gán"
assert agg.get("nearest", 0) == 0, (
    f"{agg['nearest']} keyframe phải dùng fallback 'nearest' — keyframe rơi ngoài mọi scene, "
    "kiểm tra lại scenes.json của các video đó trước khi dùng map")
print("\nOK")

cách khớp:
  frame      578528  (100.00%)

keyframe không có scene : 0
keyframe ngoài range    : 0
scene rỗng              : 0

OK


## Xem thử 1 video

In [24]:
if all_maps:
    m = all_maps[0]
    print(f"{m['group']}/{m['video_id']}: {m['num_keyframes']} keyframe, {m['num_scenes']} scene\n")

    print("keyframes[:3]:")
    print(json.dumps(m["keyframes"][:3], ensure_ascii=False, indent=2))

    print("\nscenes[0] (cắt bớt danh sách keyframes):")
    s0 = dict(m["scenes"][0])
    s0["keyframes"] = s0["keyframes"][:8] + (["..."] if len(s0["keyframes"]) > 8 else [])
    print(json.dumps(s0, ensure_ascii=False, indent=2))

    print("\nphân bố keyframe/scene (5 scene đầu):")
    for sc in m["scenes"][:5]:
        print(f"  scene {sc['scene_id']:3d}  frame [{sc['start_frame']:6d}, {sc['end_frame']:6d}]  "
              f"{sc['start_time']:7.2f}s -> {sc['end_time']:7.2f}s  {sc['num_keyframes']:3d} keyframe")
else:
    print("chưa có map nào — xem lại cell chạy")

L21_a/L21_V001: 2350 keyframe, 245 scene

keyframes[:3]:
[
  {
    "keyframe_id": "L21_V001#000000",
    "frame": 0,
    "timestamp": 0.0,
    "keyframe_file": "000000.webp",
    "keyframe_url": "https://aic-bucket-2026.s3.ap-southeast-1.amazonaws.com/Keyframes_L21_a/keyframes/L21_V001/000000.webp",
    "shot_id": 0,
    "scene_id": 0,
    "match": "frame"
  },
  {
    "keyframe_id": "L21_V001#000001",
    "frame": 1,
    "timestamp": 0.033,
    "keyframe_file": "000001.webp",
    "keyframe_url": "https://aic-bucket-2026.s3.ap-southeast-1.amazonaws.com/Keyframes_L21_a/keyframes/L21_V001/000001.webp",
    "shot_id": 0,
    "scene_id": 0,
    "match": "frame"
  },
  {
    "keyframe_id": "L21_V001#000002",
    "frame": 2,
    "timestamp": 0.067,
    "keyframe_file": "000002.webp",
    "keyframe_url": "https://aic-bucket-2026.s3.ap-southeast-1.amazonaws.com/Keyframes_L21_a/keyframes/L21_V001/000002.webp",
    "shot_id": 0,
    "scene_id": 0,
    "match": "frame"
  }
]

scenes[0] (cắt bớt d

## Ghi file

Một file / group là mặc định vì file gộp ở quy mô toàn bộ corpus dễ lên hàng trăm MB,
mở lại bằng `json.load` sẽ ngốn RAM gấp mấy lần dung lượng đĩa. File gộp vẫn ghi khi
`MERGE_ALL=True` cho tiện lúc thử nghiệm; bản `.csv` phẳng mới là thứ nên nạp cho bước
index (đọc theo dòng, không phải nuốt cả cây JSON).

In [25]:
os.makedirs(MAP_ROOT, exist_ok=True)
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
written = []


def _meta(groups, videos_maps):
    return {
        "version": 1,
        "created_at": now,
        "source": SOURCE,
        "s3_url_base": S3_URL_BASE,
        "groups": list(groups),
        "num_videos": len(videos_maps),
        "num_keyframes": sum(m["num_keyframes"] for m in videos_maps),
        "num_scenes": sum(m["num_scenes"] for m in videos_maps),
    }


for group, vids in maps.items():
    if not vids:
        continue
    doc = _meta([group], list(vids.values()))
    doc["videos"] = vids
    path = dump_json(os.path.join(MAP_ROOT, map_key(group)), doc)
    written.append(path)
    print(f"  {path}  ({os.path.getsize(path) / 2**20:.1f} MB)")

if MERGE_ALL and all_maps:
    merged = _meta(list(maps), all_maps)
    merged["videos"] = {v: m for g in maps.values() for v, m in g.items()}
    path = dump_json(os.path.join(MAP_ROOT, "keyframe_scene_map.json"), merged)
    written.append(path)
    print(f"  {path}  ({os.path.getsize(path) / 2**20:.1f} MB)")

if WRITE_CSV and all_maps:
    path = os.path.join(MAP_ROOT, "keyframe_scene_flat.csv")
    cols = ["keyframe_id", "group", "video_id", "frame", "timestamp", "shot_id", "scene_id",
            "scene_start_frame", "scene_end_frame", "scene_start_time", "scene_end_time",
            "match", "keyframe_url", "scene_url"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for m in all_maps:          # ghi theo dòng, không dựng list trung gian
            by_id = {s["scene_id"]: s for s in m["scenes"]}
            for kf in m["keyframes"]:
                sc = by_id.get(kf["scene_id"], {})
                w.writerow([
                    kf["keyframe_id"], m["group"], m["video_id"], kf["frame"], kf["timestamp"],
                    kf["shot_id"], kf["scene_id"],
                    sc.get("start_frame"), sc.get("end_frame"),
                    sc.get("start_time"), sc.get("end_time"),
                    kf["match"], kf["keyframe_url"], sc.get("scene_url"),
                ])
    written.append(path)
    print(f"  {path}  ({os.path.getsize(path) / 2**20:.1f} MB)")

  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L21_a/maps/keyframe_scene_map.json  (22.4 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L22_a/maps/keyframe_scene_map.json  (27.0 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L23_a/maps/keyframe_scene_map.json  (2.0 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L24_a/maps/keyframe_scene_map.json  (6.1 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L26_a/maps/keyframe_scene_map.json  (22.1 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L26_b/maps/keyframe_scene_map.json  (25.8 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L26_c/maps/keyframe_scene_map.json  (15.5 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L26_d/maps/keyframe_scene_map.json  (27.7 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L26_e/maps/keyframe_scene_map.json  (29.3 MB)
  /Users/kinh.nvnamitech.io/Documents/rackfocus/maps/Maps_L27_a/ma

## Đọc lại để chắc

Ghi xong không có nghĩa là đọc lại được: JSON hỏng, encoding sai, hay ghi nửa chừng vì
hết đĩa đều chỉ lộ ra ở bước load. Cell này load lại file vừa ghi và so số đếm.

In [26]:
for path in written:
    if not path.endswith(".json"):
        continue
    with open(path, encoding="utf-8") as f:
        doc = json.load(f)
    n_kf = sum(v["num_keyframes"] for v in doc["videos"].values())
    ok = (len(doc["videos"]) == doc["num_videos"] and n_kf == doc["num_keyframes"])
    print(f"  {'OK  ' if ok else 'LỆCH'} {os.path.basename(path)}: "
          f"{doc['num_videos']} video, {doc['num_keyframes']} keyframe")
    assert ok, f"{path}: số đếm trong header không khớp nội dung"

if WRITE_CSV and all_maps:
    path = os.path.join(MAP_ROOT, "keyframe_scene_flat.csv")
    with open(path, encoding="utf-8") as f:
        rows = sum(1 for _ in f) - 1
    print(f"  {'OK  ' if rows == total_kf else 'LỆCH'} keyframe_scene_flat.csv: {rows} dòng")
    assert rows == total_kf, f"csv có {rows} dòng, mong đợi {total_kf}"

print("\ntra thử keyframe -> scene:")
if all_maps:
    m = all_maps[0]
    idx = {kf["keyframe_id"]: kf["scene_id"] for kf in m["keyframes"]}
    for kid in list(idx)[:5]:
        print(f"  {kid} -> scene {idx[kid]}")

  OK   keyframe_scene_map.json: 29 video, 57526 keyframe
  OK   keyframe_scene_map.json: 31 video, 70629 keyframe
  OK   keyframe_scene_map.json: 25 video, 5153 keyframe
  OK   keyframe_scene_map.json: 43 video, 16236 keyframe
  OK   keyframe_scene_map.json: 99 video, 58902 keyframe
  OK   keyframe_scene_map.json: 100 video, 65830 keyframe
  OK   keyframe_scene_map.json: 53 video, 38605 keyframe
  OK   keyframe_scene_map.json: 100 video, 73683 keyframe
  OK   keyframe_scene_map.json: 99 video, 73790 keyframe
  OK   keyframe_scene_map.json: 16 video, 18865 keyframe
  OK   keyframe_scene_map.json: 24 video, 36137 keyframe
  OK   keyframe_scene_map.json: 23 video, 33275 keyframe
  OK   keyframe_scene_map.json: 96 video, 29897 keyframe
  OK   keyframe_scene_map.json: 738 video, 578528 keyframe
  OK   keyframe_scene_flat.csv: 578528 dòng

tra thử keyframe -> scene:
  L21_V001#000000 -> scene 0
  L21_V001#000001 -> scene 0
  L21_V001#000002 -> scene 0
  L21_V001#000004 -> scene 0
  L21_V001#

## Upload map lên S3 (tuỳ chọn)

Đặt `UPLOAD_MAP = True` ở cell config. Key giữ nguyên `Maps_<group>/maps/...` để cùng cây
với `Keyframes_*` / `Keyscence_*`.

In [27]:
if not UPLOAD_MAP:
    print("UPLOAD_MAP=False — chỉ ghi local. Bật lên rồi chạy lại cell này nếu cần đẩy S3.")
elif not written:
    print("chưa ghi được file nào.")
else:
    import boto3

    _cli = getattr(reader, "client", None) or boto3.client(
        "s3",
        aws_access_key_id=os.environ.get("AWS_ACCESS_KEY") or None,
        aws_secret_access_key=os.environ.get("AWS_SECRET_KEY") or None,
        region_name=os.environ["AWS_REGION"],
    )
    bucket = os.environ["AWS_BUCKET_NAME"]

    for path in written:
        key = os.path.relpath(path, MAP_ROOT).replace(os.sep, "/")
        if "/" not in key:                      # file gộp / csv nằm ở gốc MAP_ROOT
            key = f"{MAP_PREFIX}all/{MAP_SUB}/{key}"
        _cli.upload_file(path, bucket, key)
        print(f"  {key}  -> {url_of(key)}")

UPLOAD_MAP=False — chỉ ghi local. Bật lên rồi chạy lại cell này nếu cần đẩy S3.
